# Multimodal Vectorless RAG with LangGraph + NullVector

This cookbook is the permanent framework-backed LangGraph walkthrough for the current NullVector runtime.
It uses the acquisition manifest and tree manifest as the only artifact entrypoints, routes optional live text through `GatewayService`, keeps visual enrichment attachment-only, and blocks polluted trees with a structured failure instead of fabricating grounded answers.


### Environment

- Default mode is fixture-first and uses `fixtures/pdfs/phase01/born_digital_with_outline.pdf`.
- Optional local mode can be enabled with `USE_LOCAL_PDF = True`; if the resulting tree is polluted, the notebook blocks answer generation honestly.
- All live text generation goes through NullVector `GatewayService` with LiteLLM/OpenRouter settings from the real environment.
- Multimodal enrichment stays attachment-only through the NullVector multimodal gateway and a deterministic noop provider.
- Validation for this notebook is run with deprecation warnings treated as errors.


In [1]:
# environment setup
from pathlib import Path
import hashlib
import json
import os
import re
import warnings
from typing import Any, Literal, TypedDict

warnings.filterwarnings("error", category=DeprecationWarning)

REPO_ROOT = Path.cwd()
COOKBOOK_ROOT = REPO_ROOT / "cookbook"
COOKBOOK_RUNTIME_ROOT = (COOKBOOK_ROOT / "artifacts" / "langgraph_rag").resolve()
COOKBOOK_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

USE_LOCAL_PDF = False
LOCAL_PDF = REPO_ROOT / "903000608.pdf"
FIXTURE_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"

if USE_LOCAL_PDF:
    if not LOCAL_PDF.exists():
        raise FileNotFoundError(f"USE_LOCAL_PDF=True but local PDF is missing: {LOCAL_PDF}")
    PDF_PATH = LOCAL_PDF
    PDF_SOURCE_MODE = "local_real_pdf"
else:
    if not FIXTURE_PDF.exists():
        raise FileNotFoundError(f"Fixture PDF is missing: {FIXTURE_PDF}")
    PDF_PATH = FIXTURE_PDF
    PDF_SOURCE_MODE = "fixture_pdf"

print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "pdf_path": str(PDF_PATH),
    "pdf_source_mode": PDF_SOURCE_MODE,
    "cookbook_runtime_root": str(COOKBOOK_RUNTIME_ROOT),
    "use_local_pdf": USE_LOCAL_PDF,
}, indent=2, ensure_ascii=True))


{
  "repo_root": "/home/pruthvi/projects/NullVector",
  "pdf_path": "/home/pruthvi/projects/NullVector/fixtures/pdfs/phase01/born_digital_with_outline.pdf",
  "pdf_source_mode": "fixture_pdf",
  "cookbook_runtime_root": "/home/pruthvi/projects/NullVector/cookbook/artifacts/langgraph_rag",
  "use_local_pdf": false
}


In [2]:
# imports
from pydantic import BaseModel, Field, TypeAdapter
from langgraph.graph import END, START, StateGraph

from nullvector.domain import (
    AcquisitionRequest,
    AcquisitionRunManifest,
    AcquisitionSettings,
    CanonicalDocumentLedger,
    NodeCard,
    NodeSummary,
    StructuredRegionInsight,
    TreeBuildManifest,
    TreeBuildRequest,
    TreeSettings,
    UnresolvedRegion,
    VisualArtifact,
    VisualEnrichmentRequest,
    VisualRegionReference,
)
from nullvector.ingest import acquire_document
from nullvector.ingest.acquisition_artifacts import settings_digest as acquisition_settings_digest
from nullvector.ingest.fingerprint import fingerprint_document
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayError,
    GatewayRequest,
    GatewayService,
    LiteLLMProviderConfig,
    LLMMessage,
    LLMRole,
    MultimodalGatewayConfig,
    MultimodalGatewayError,
    MultimodalGatewayService,
    MultimodalProviderConfig,
    NoopMultimodalProviderAdapter,
    NoopMultimodalResponse,
    VisualEnrichmentService,
)
from nullvector.tree import build_tree

print("✓ LangGraph + NullVector imports succeeded")


✓ LangGraph + NullVector imports succeeded


In [3]:
# configuration
class NotebookAnswerResponse(BaseModel):
    answer: str


class TreeQualityReport(BaseModel):
    status: Literal["accepted", "rejected"]
    tree_manifest_path: str
    page_count: int
    node_count: int
    max_allowed_nodes: int
    suspicious_node_count: int
    suspicious_ratio: float
    suspicious_one_page_ratio: float
    average_alpha_ratio: float
    issue_codes: tuple[str, ...] = ()
    sample_rejected_titles: tuple[str, ...] = ()


class CookbookFailure(BaseModel):
    status: Literal["blocked_tree_quality"] = "blocked_tree_quality"
    reason_code: Literal["rejected_tree_quality"] = "rejected_tree_quality"
    message: str
    tree_manifest_path: str
    issue_codes: tuple[str, ...] = ()
    sample_rejected_titles: tuple[str, ...] = ()
    metrics: dict[str, Any] = Field(default_factory=dict)


TreeQualityReport.model_rebuild()
CookbookFailure.model_rebuild()


NODE_CARD_ADAPTER = TypeAdapter(tuple[NodeCard, ...])
NODE_SUMMARY_ADAPTER = TypeAdapter(tuple[NodeSummary, ...])
NOISE_TOKENS = {
    "page",
    "offset",
    "invoice",
    "subtotal",
    "total",
    "gst",
    "qty",
    "amount",
    "mrp",
    "price",
    "phone",
    "email",
    "fax",
    "www",
    "date",
    "rs",
}
NUMERIC_OR_PUNCT_RE = re.compile(r"^[\W\d_]+$")


def load_typed_json(path: str | Path, adapter: TypeAdapter):
    return adapter.validate_json(Path(path).read_text(encoding="utf-8"))


def stable_digest(payload: dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")
    ).hexdigest()


def tree_settings_digest(settings: TreeSettings) -> str:
    return stable_digest(settings.model_dump(mode="json"))


def choose_visual_attachment_path(region: VisualRegionReference) -> tuple[str | None, str | None]:
    if region.asset_path:
        return region.asset_path, "asset_path"
    if region.page_render_path:
        return region.page_render_path, "page_render_path"
    return None, None


def block_to_region_reference(document_id: str, page_index: int, block: VisualArtifact | UnresolvedRegion) -> VisualRegionReference:
    region_id = block.visual_id if isinstance(block, VisualArtifact) else block.region_id
    image_ref = block.image_ref if isinstance(block, VisualArtifact) else None
    return VisualRegionReference(
        document_id=document_id,
        page_index=page_index,
        region_id=region_id,
        bbox=block.bbox,
        image_ref=image_ref,
        asset_path=block.asset_path,
        page_render_path=block.page_render_path,
        render_dpi=block.render_dpi,
        coordinate_space=block.coordinate_space,
    )


def collect_enrichable_regions(ledger: CanonicalDocumentLedger) -> list[dict[str, Any]]:
    regions: list[dict[str, Any]] = []
    for page in ledger.pages:
        for block in page.blocks:
            if isinstance(block, VisualArtifact):
                if not block.needs_enrichment:
                    continue
                region = block_to_region_reference(ledger.document_id, page.page_index, block)
                attachment_path, attachment_kind = choose_visual_attachment_path(region)
                regions.append({
                    "page_index": page.page_index,
                    "region": region,
                    "attachment_path": attachment_path,
                    "attachment_kind": attachment_kind,
                    "kind": "visual_artifact",
                    "reason_code": None,
                    "kind_hint": block.kind_hint,
                })
            elif isinstance(block, UnresolvedRegion):
                region = block_to_region_reference(ledger.document_id, page.page_index, block)
                attachment_path, attachment_kind = choose_visual_attachment_path(region)
                regions.append({
                    "page_index": page.page_index,
                    "region": region,
                    "attachment_path": attachment_path,
                    "attachment_kind": attachment_kind,
                    "kind": "unresolved_region",
                    "reason_code": block.reason_code,
                    "kind_hint": None,
                })
    return regions


def _title_tokens(title: str) -> set[str]:
    return {token for token in re.split(r"[^A-Za-z0-9]+", title.casefold()) if token}


def _title_metrics(title: str) -> tuple[float, tuple[str, ...]]:
    stripped = title.strip()
    alnum = [char for char in stripped if char.isalnum()]
    alpha = [char for char in stripped if char.isalpha()]
    digits = [char for char in stripped if char.isdigit()]
    alpha_ratio = len(alpha) / max(1, len(alnum))
    digit_ratio = len(digits) / max(1, len(alnum))
    issue_codes: list[str] = []
    tokens = _title_tokens(stripped)
    if NUMERIC_OR_PUNCT_RE.fullmatch(stripped):
        issue_codes.append("numeric_or_punct_only_title")
    if stripped.casefold().startswith("page ") and any(char.isdigit() for char in stripped):
        issue_codes.append("page_label_title")
    if tokens & NOISE_TOKENS:
        issue_codes.append("noise_token_title")
    if alpha_ratio < 0.45:
        issue_codes.append("low_alpha_ratio_title")
    if digit_ratio > 0.45:
        issue_codes.append("digit_heavy_title")
    return alpha_ratio, tuple(dict.fromkeys(issue_codes))


def evaluate_tree_quality(
    node_cards: tuple[NodeCard, ...],
    *,
    tree_manifest_path: str,
    page_count: int,
) -> TreeQualityReport:
    max_allowed_nodes = max(24, min(400, page_count * 6))
    suspicious_titles: list[str] = []
    suspicious_node_count = 0
    suspicious_one_page_count = 0
    alpha_ratios: list[float] = []

    for card in node_cards:
        alpha_ratio, title_issue_codes = _title_metrics(card.title)
        alpha_ratios.append(alpha_ratio)
        page_span_length = card.page_span.end_page - card.page_span.start_page + 1
        if title_issue_codes:
            suspicious_node_count += 1
            suspicious_titles.append(card.title)
            if page_span_length == 1:
                suspicious_one_page_count += 1

    node_count = len(node_cards)
    suspicious_ratio = suspicious_node_count / max(1, node_count)
    suspicious_one_page_ratio = suspicious_one_page_count / max(1, node_count)
    average_alpha_ratio = sum(alpha_ratios) / max(1, len(alpha_ratios))

    issue_codes: list[str] = []
    if node_count > max_allowed_nodes:
        issue_codes.append("excessive_committed_node_count")
    if suspicious_ratio > 0.18:
        issue_codes.append("suspicious_title_ratio")
    if suspicious_one_page_ratio > 0.10:
        issue_codes.append("suspicious_one_page_ratio")
    if average_alpha_ratio < 0.72:
        issue_codes.append("low_alphabetic_title_ratio")

    return TreeQualityReport(
        status="rejected" if issue_codes else "accepted",
        tree_manifest_path=tree_manifest_path,
        page_count=page_count,
        node_count=node_count,
        max_allowed_nodes=max_allowed_nodes,
        suspicious_node_count=suspicious_node_count,
        suspicious_ratio=round(suspicious_ratio, 3),
        suspicious_one_page_ratio=round(suspicious_one_page_ratio, 3),
        average_alpha_ratio=round(average_alpha_ratio, 3),
        issue_codes=tuple(issue_codes),
        sample_rejected_titles=tuple(suspicious_titles[:10]),
    )


def build_tree_quality_failure(*, tree_manifest_path: str, quality_report: TreeQualityReport) -> CookbookFailure:
    return CookbookFailure(
        message="Tree artifacts are not cookbook-grade for retrieval; answer generation was blocked.",
        tree_manifest_path=tree_manifest_path,
        issue_codes=quality_report.issue_codes,
        sample_rejected_titles=quality_report.sample_rejected_titles,
        metrics={
            "page_count": quality_report.page_count,
            "node_count": quality_report.node_count,
            "max_allowed_nodes": quality_report.max_allowed_nodes,
            "suspicious_node_count": quality_report.suspicious_node_count,
            "suspicious_ratio": quality_report.suspicious_ratio,
            "suspicious_one_page_ratio": quality_report.suspicious_one_page_ratio,
            "average_alpha_ratio": quality_report.average_alpha_ratio,
        },
    )


def format_grounded_fallback_answer(
    *,
    query: str,
    context_chunks: list[dict[str, Any]],
    visual_trace: list[str],
) -> str:
    lines = [f"Query: {query}", "", "Grounded answer assembled from committed NullVector nodes:"]
    if not context_chunks:
        lines.extend(["", "No relevant committed nodes were available."])
        return "\n".join(lines)
    for index, chunk in enumerate(context_chunks, start=1):
        page_span = chunk["page_span"]
        lines.append(
            f"{index}. {chunk['title']} [pages {page_span['start_page']}-{page_span['end_page']}]"
        )
        lines.append(f"   {chunk['summary']}")
        if chunk.get("visual_summaries"):
            lines.append("   Visual context:")
            for item in chunk["visual_summaries"]:
                lines.append(f"   - {item}")
        lines.append(f"   Citation: node_id={chunk['node_id']}")
        lines.append("")
    if visual_trace:
        lines.append("Visual trace:")
        for item in visual_trace:
            lines.append(f"- {item}")
    return "\n".join(lines).strip()


def resolve_current_cookbook_runs(pdf_path: Path) -> dict[str, Any]:
    fingerprint = fingerprint_document(str(pdf_path))
    acquisition_settings = AcquisitionSettings()
    tree_settings = TreeSettings()
    acquisition_suffix = f"{fingerprint.sha256[:12]}-{acquisition_settings_digest(acquisition_settings)[:8]}"
    tree_suffix = f"{fingerprint.sha256[:12]}-{tree_settings_digest(tree_settings)[:8]}"

    acquisition_root = (COOKBOOK_RUNTIME_ROOT / "acquisition_runs").resolve()
    acquisition_root.mkdir(parents=True, exist_ok=True)
    acquisition_run_id = f"langgraph-rag-acquisition-{acquisition_suffix}"
    tree_run_id = f"langgraph-rag-tree-{tree_suffix}"

    acquired = acquire_document(
        AcquisitionRequest(
            source_path=str(pdf_path),
            acquisition_run_id=acquisition_run_id,
            artifact_root=str(acquisition_root),
            settings=acquisition_settings,
        )
    )
    acquisition_manifest_path = Path(acquired.artifact_root) / "manifest.json"
    acquisition_manifest = AcquisitionRunManifest.model_validate_json(
        acquisition_manifest_path.read_text(encoding="utf-8")
    )
    ledger = CanonicalDocumentLedger.model_validate_json(
        Path(acquisition_manifest.ledger_path).read_text(encoding="utf-8")
    )

    built = build_tree(
        TreeBuildRequest(
            acquisition_manifest_path=str(acquisition_manifest_path),
            tree_run_id=tree_run_id,
            settings=tree_settings,
        )
    )
    tree_manifest_path = Path(built.artifact_root) / "manifest.json"
    tree_manifest = TreeBuildManifest.model_validate_json(
        tree_manifest_path.read_text(encoding="utf-8")
    )
    node_cards = load_typed_json(tree_manifest.node_cards_path, NODE_CARD_ADAPTER)
    summaries_by_id = {}
    if tree_manifest.node_summaries_path:
        for summary in load_typed_json(tree_manifest.node_summaries_path, NODE_SUMMARY_ADAPTER):
            summaries_by_id[summary.node_id] = summary

    visual_regions = collect_enrichable_regions(ledger)
    return {
        "fingerprint": fingerprint,
        "acquisition_manifest": acquisition_manifest,
        "acquisition_manifest_path": acquisition_manifest_path,
        "tree_manifest": tree_manifest,
        "tree_manifest_path": tree_manifest_path,
        "ledger": ledger,
        "node_cards": node_cards,
        "summaries_by_id": summaries_by_id,
        "visual_regions": visual_regions,
        "acquisition_run_id": acquisition_run_id,
        "tree_run_id": tree_run_id,
    }


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openrouter/openai/gpt-4.1-mini")
TEXT_LLM_ENABLED = bool(OPENROUTER_API_KEY)


In [4]:
# execution
runtime = resolve_current_cookbook_runs(PDF_PATH)
fingerprint = runtime["fingerprint"]
acquisition_manifest = runtime["acquisition_manifest"]
tree_manifest = runtime["tree_manifest"]
tree_manifest_path = str(runtime["tree_manifest_path"])
canonical_ledger = runtime["ledger"]
node_cards = runtime["node_cards"]
summaries_by_id = runtime["summaries_by_id"]
visual_regions = runtime["visual_regions"]

quality_report = evaluate_tree_quality(
    node_cards,
    tree_manifest_path=tree_manifest_path,
    page_count=acquisition_manifest.page_count,
)
cookbook_failure = (
    build_tree_quality_failure(tree_manifest_path=tree_manifest_path, quality_report=quality_report)
    if quality_report.status == "rejected"
    else None
)

print(json.dumps({
    "document_id": fingerprint.document_id,
    "acquisition_run_id": acquisition_manifest.acquisition_run_id,
    "tree_run_id": tree_manifest.tree_run_id,
    "page_count": acquisition_manifest.page_count,
    "node_count": len(node_cards),
    "quality_status": quality_report.status,
    "quality_issue_codes": list(quality_report.issue_codes),
    "pdf_source_mode": PDF_SOURCE_MODE,
}, indent=2, ensure_ascii=True))


{
  "document_id": "015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52",
  "acquisition_run_id": "langgraph-rag-acquisition-015e3d97fc52-370e1038",
  "tree_run_id": "langgraph-rag-tree-015e3d97fc52-42c8480b",
  "page_count": 3,
  "node_count": 2,
  "quality_status": "accepted",
  "quality_issue_codes": [],
  "pdf_source_mode": "fixture_pdf"
}


In [5]:
# execution
def region_owner(region_page_index: int, cards: tuple[NodeCard, ...]) -> NodeCard | None:
    candidates = [
        card for card in cards
        if card.page_span.start_page <= region_page_index <= card.page_span.end_page
    ]
    if not candidates:
        return None
    return sorted(
        candidates,
        key=lambda card: (
            -card.level,
            (card.page_span.end_page - card.page_span.start_page),
            len(card.path),
            card.node_id,
        ),
    )[0]


node_visual_regions: dict[str, list[dict[str, Any]]] = {}
node_details: dict[str, dict[str, Any]] = {}
if cookbook_failure is None:
    for item in visual_regions:
        if item["attachment_path"] is None:
            continue
        owner = region_owner(item["page_index"], node_cards)
        if owner is None:
            continue
        item["region"] = item["region"].model_copy(update={"node_id": owner.node_id})
        node_visual_regions.setdefault(owner.node_id, []).append(item)

    for card in node_cards:
        summary = summaries_by_id.get(card.node_id)
        summary_text = summary.summary if summary is not None else (card.summary or card.title)
        node_details[card.node_id] = {
            "node_id": card.node_id,
            "title": card.title,
            "summary": summary_text,
            "keywords": list(summary.keywords) if summary is not None else list(card.keywords),
            "page_span": {
                "start_page": card.page_span.start_page,
                "end_page": card.page_span.end_page,
            },
            "visual_regions": node_visual_regions.get(card.node_id, []),
        }

print(json.dumps({
    "visual_region_count": len(visual_regions),
    "asset_backed_region_count": sum(1 for item in visual_regions if item["attachment_path"] is not None),
    "nodes_with_visual_regions": sum(1 for regions in node_visual_regions.values() if regions),
    "blocked_tree_quality": cookbook_failure is not None,
}, indent=2, ensure_ascii=True))


{
  "visual_region_count": 0,
  "asset_backed_region_count": 0,
  "nodes_with_visual_regions": 0,
  "blocked_tree_quality": false
}


In [6]:
# execution
multimodal_gateway = MultimodalGatewayService(
    MultimodalGatewayConfig(
        provider=MultimodalProviderConfig(model="cookbook-multimodal-noop"),
        audit_root=str(COOKBOOK_RUNTIME_ROOT / "multimodal-audit"),
    ),
    provider_adapter=NoopMultimodalProviderAdapter(
        {
            "visual_region_enrichment": NoopMultimodalResponse(
                output_json={
                    "insight": {
                        "summary": "Attachment-backed visual region reviewed through the NullVector multimodal gateway.",
                        "labels": ["asset-backed-region", "attachment-only"],
                        "attributes": {"demo": True},
                        "confidence": 0.5,
                    }
                }
            )
        }
    ),
)
visual_enrichment_service = VisualEnrichmentService(multimodal_gateway)

text_gateway = None
if TEXT_LLM_ENABLED:
    text_gateway = GatewayService(
        GatewayConfig(
            provider=LiteLLMProviderConfig(
                model=OPENROUTER_MODEL,
                api_key=OPENROUTER_API_KEY,
                api_base=OPENROUTER_API_BASE,
            ),
            audit=GatewayAuditConfig(persist_root=str(COOKBOOK_RUNTIME_ROOT / "llm-audit")),
        )
    )

print(json.dumps({
    "text_llm_enabled": TEXT_LLM_ENABLED,
    "openrouter_model": OPENROUTER_MODEL,
    "openrouter_api_base": OPENROUTER_API_BASE,
    "multimodal_provider": "noop",
}, indent=2, ensure_ascii=True))


{
  "text_llm_enabled": false,
  "openrouter_model": "openrouter/openai/gpt-4.1-mini",
  "openrouter_api_base": "https://openrouter.ai/api/v1",
  "multimodal_provider": "noop"
}


In [7]:
# execution

def token_set(text: str) -> set[str]:
    return {token for token in ''.join(ch.lower() if ch.isalnum() else ' ' for ch in text).split() if token}


class RAGState(TypedDict, total=False):
    query: str
    node_details: dict[str, dict[str, Any]]
    selected_node_ids: list[str]
    retrieval_reasoning: str
    retrieval_trace: list[str]
    visual_attachments_by_node: dict[str, list[dict[str, Any]]]
    context_chunks: list[dict[str, Any]]
    answer: str
    answer_mode: str
    live_text_result: dict[str, Any]


def load_artifacts_node(state: RAGState) -> dict[str, Any]:
    return {
        "node_details": node_details,
        "retrieval_trace": ["Loaded current acquisition and tree manifests only."],
    }


def select_sections_node(state: RAGState) -> dict[str, Any]:
    query_tokens = token_set(state["query"])
    scored: list[tuple[int, int, int, str]] = []
    for node_id, detail in state["node_details"].items():
        searchable = ' '.join([
            detail["title"],
            detail["summary"],
            ' '.join(detail["keywords"]),
        ])
        overlap = len(query_tokens & token_set(searchable))
        page_span = detail["page_span"]
        span_length = page_span["end_page"] - page_span["start_page"]
        scored.append((overlap, -page_span["start_page"], -span_length, node_id))
    if any(item[0] > 0 for item in scored):
        ordered_ids = [item[3] for item in sorted(scored, reverse=True)]
        reasoning = "Selected committed nodes by lexical overlap over title, summary, and keywords."
    else:
        ordered_ids = sorted(
            state["node_details"],
            key=lambda node_id: (
                state["node_details"][node_id]["page_span"]["start_page"],
                state["node_details"][node_id]["page_span"]["end_page"],
                node_id,
            ),
        )
        reasoning = "No lexical overlap; selected earliest committed nodes deterministically."
    selected_node_ids = ordered_ids[:2]
    return {
        "selected_node_ids": selected_node_ids,
        "retrieval_reasoning": reasoning,
        "retrieval_trace": [*state["retrieval_trace"], reasoning],
    }


def enrich_visuals_node(state: RAGState) -> dict[str, Any]:
    attachments_by_node: dict[str, list[dict[str, Any]]] = {}
    trace = list(state["retrieval_trace"])
    for node_id in state["selected_node_ids"]:
        regions = node_visual_regions.get(node_id, [])
        if not regions:
            continue
        for index, region_item in enumerate(regions[:1]):
            try:
                attachment = visual_enrichment_service.enrich(
                    VisualEnrichmentRequest(
                        request_id=f"cookbook-visual-{node_id}-{index}",
                        region=region_item["region"],
                        prompt="Describe this visual region briefly and conservatively.",
                        node_id=node_id,
                        metadata={"notebook": "langgraph_rag_cookbook"},
                    )
                )
            except MultimodalGatewayError as exc:
                trace.append(f"Visual enrichment failed for {node_id}: {exc.failure.message}")
                continue
            attachments_by_node.setdefault(node_id, []).append(
                {
                    "summary": attachment.insight.summary,
                    "labels": list(attachment.insight.labels),
                    "audit_path": attachment.audit_path,
                    "provider_identity": attachment.provider_identity,
                }
            )
    if not attachments_by_node:
        trace.append("No asset-backed visual regions available for the selected nodes.")
    else:
        trace.append("Attached multimodal summaries for selected asset-backed visual regions.")
    return {
        "visual_attachments_by_node": attachments_by_node,
        "retrieval_trace": trace,
    }


def assemble_context_node(state: RAGState) -> dict[str, Any]:
    chunks: list[dict[str, Any]] = []
    for node_id in state["selected_node_ids"]:
        detail = state["node_details"][node_id]
        attachments = state["visual_attachments_by_node"].get(node_id, [])
        chunks.append(
            {
                "node_id": node_id,
                "title": detail["title"],
                "summary": detail["summary"],
                "page_span": detail["page_span"],
                "visual_summaries": [item["summary"] for item in attachments],
            }
        )
    return {"context_chunks": chunks}


def generate_answer_node(state: RAGState) -> dict[str, Any]:
    visual_trace = [item for item in state["retrieval_trace"] if "visual" in item.casefold()]
    if text_gateway is None:
        return {
            "answer": format_grounded_fallback_answer(
                query=state["query"],
                context_chunks=state["context_chunks"],
                visual_trace=visual_trace,
            ),
            "answer_mode": "deterministic_fallback",
            "live_text_result": {
                "status": "skipped",
                "reason": "OPENROUTER_API_KEY not configured",
            },
        }

    context_lines: list[str] = []
    for chunk in state["context_chunks"]:
        context_lines.append(
            f"Title: {chunk['title']} | Pages: {chunk['page_span']['start_page']}-{chunk['page_span']['end_page']}"
        )
        context_lines.append(f"Summary: {chunk['summary']}")
        if chunk["visual_summaries"]:
            context_lines.append("Visual: " + " | ".join(chunk["visual_summaries"]))
        context_lines.append("")
    try:
        response = text_gateway.invoke(
            GatewayRequest[NotebookAnswerResponse](
                operation_name="cookbook_answer_generation",
                messages=(
                    LLMMessage(
                        role=LLMRole.SYSTEM,
                        content=(
                            "Answer using only the provided cookbook context. "
                            "If the context is insufficient, say so plainly."
                        ),
                    ),
                    LLMMessage(
                        role=LLMRole.USER,
                        content=f"Query: {state['query']}\n\nContext:\n" + "\n".join(context_lines),
                    ),
                ),
                response_model=NotebookAnswerResponse,
                max_output_tokens=300,
            )
        )
        return {
            "answer": response.output.answer,
            "answer_mode": "gateway_live_text",
            "live_text_result": {
                "status": "success",
                "provider_name": response.provider_name,
                "model_name": response.model_name,
                "assurance_mode": response.assurance_mode.value,
                "audit_path": response.audit_path,
            },
        }
    except GatewayError as exc:
        return {
            "answer": format_grounded_fallback_answer(
                query=state["query"],
                context_chunks=state["context_chunks"],
                visual_trace=visual_trace,
            ),
            "answer_mode": "deterministic_fallback",
            "live_text_result": {
                "status": "failure",
                "category": exc.failure.category.value,
                "message": exc.failure.message,
                "audit_path": exc.audit_path,
            },
        }


rag_graph = None
if cookbook_failure is None:
    graph = StateGraph(RAGState)
    graph.add_node("load_artifacts", load_artifacts_node)
    graph.add_node("select_sections", select_sections_node)
    graph.add_node("enrich_visuals", enrich_visuals_node)
    graph.add_node("assemble_context", assemble_context_node)
    graph.add_node("generate_answer", generate_answer_node)
    graph.add_edge(START, "load_artifacts")
    graph.add_edge("load_artifacts", "select_sections")
    graph.add_edge("select_sections", "enrich_visuals")
    graph.add_edge("enrich_visuals", "assemble_context")
    graph.add_edge("assemble_context", "generate_answer")
    graph.add_edge("generate_answer", END)
    rag_graph = graph.compile()

print(json.dumps({
    "graph_ready": rag_graph is not None,
    "text_llm_enabled": TEXT_LLM_ENABLED,
}, indent=2, ensure_ascii=True))


{
  "graph_ready": true,
  "text_llm_enabled": false
}


In [8]:
# execution
if cookbook_failure is None:
    result = rag_graph.invoke({"query": "what is on first page?"})
    result["status"] = "ready"
else:
    result = {
        "status": "blocked_tree_quality",
        "failure": cookbook_failure.model_dump(mode="json"),
        "answer": None,
        "answer_mode": None,
        "live_text_result": {
            "status": "skipped",
            "reason": "tree quality gate rejected the current tree artifacts",
        },
    }

if text_gateway is not None:
    text_gateway.close()

print(json.dumps(result, indent=2, ensure_ascii=True))


{
  "query": "what is on first page?",
  "node_details": {
    "20b912e91b26b030e990db21ab91e02669544b4a2851208b13b4b7abb9aa8e47": {
      "node_id": "20b912e91b26b030e990db21ab91e02669544b4a2851208b13b4b7abb9aa8e47",
      "title": "Overview",
      "summary": "Overview",
      "keywords": [],
      "page_span": {
        "start_page": 0,
        "end_page": 1
      },
      "visual_regions": []
    },
    "c26b909876af1a23d7907bc2585952ca5c931496820e9ae0ab51d19765bd896a": {
      "node_id": "c26b909876af1a23d7907bc2585952ca5c931496820e9ae0ab51d19765bd896a",
      "title": "Appendix",
      "summary": "Appendix",
      "keywords": [],
      "page_span": {
        "start_page": 2,
        "end_page": 2
      },
      "visual_regions": []
    }
  },
  "selected_node_ids": [
    "20b912e91b26b030e990db21ab91e02669544b4a2851208b13b4b7abb9aa8e47",
    "c26b909876af1a23d7907bc2585952ca5c931496820e9ae0ab51d19765bd896a"
  ],
  "retrieval_reasoning": "No lexical overlap; selected earliest comm

In [9]:
# inspect results
multimodal_audit_root = COOKBOOK_RUNTIME_ROOT / "multimodal-audit"
llm_audit_root = COOKBOOK_RUNTIME_ROOT / "llm-audit"
multimodal_audit_files = sorted(multimodal_audit_root.glob("*.json")) if multimodal_audit_root.exists() else []
llm_audit_files = sorted(llm_audit_root.glob("*.json")) if llm_audit_root.exists() else []
asset_backed_region_count = sum(1 for item in visual_regions if item["attachment_path"] is not None)
attachment_count = sum(len(items) for items in node_visual_regions.values()) if cookbook_failure is None else 0

summary = {
    "pdf": {
        "path": str(PDF_PATH),
        "source_mode": PDF_SOURCE_MODE,
        "document_id": fingerprint.document_id,
    },
    "artifacts": {
        "acquisition_manifest_path": str(runtime["acquisition_manifest_path"]),
        "tree_manifest_path": tree_manifest_path,
        "acquisition_run_id": acquisition_manifest.acquisition_run_id,
        "tree_run_id": tree_manifest.tree_run_id,
    },
    "tree_quality": quality_report.model_dump(mode="json"),
    "graph": {
        "status": result["status"],
        "answer_mode": result.get("answer_mode"),
        "selected_node_ids": result.get("selected_node_ids", []),
        "retrieval_reasoning": result.get("retrieval_reasoning"),
        "retrieval_trace": result.get("retrieval_trace", []),
    },
    "visual": {
        "region_count": len(visual_regions),
        "asset_backed_region_count": asset_backed_region_count,
        "attachment_count": attachment_count,
        "multimodal_audit_count": len(multimodal_audit_files),
    },
    "live_text": result.get("live_text_result", {"status": "unknown"}),
    "failure": result.get("failure"),
    "answer_excerpt": (result.get("answer") or "")[:500],
}
print(json.dumps(summary, indent=2, ensure_ascii=True))


{
  "pdf": {
    "path": "/home/pruthvi/projects/NullVector/fixtures/pdfs/phase01/born_digital_with_outline.pdf",
    "source_mode": "fixture_pdf",
    "document_id": "015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52"
  },
  "artifacts": {
    "acquisition_manifest_path": "/home/pruthvi/projects/NullVector/cookbook/artifacts/langgraph_rag/acquisition_runs/langgraph-rag-acquisition-015e3d97fc52-370e1038/015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52/manifest.json",
    "tree_manifest_path": "/home/pruthvi/projects/NullVector/cookbook/artifacts/langgraph_rag/acquisition_runs/langgraph-rag-acquisition-015e3d97fc52-370e1038/015e3d97fc52d47cf609c750ced10fb5f47ad21f1e1601b28dd55ab1b4a54b52/tree/langgraph-rag-tree-015e3d97fc52-42c8480b/manifest.json",
    "acquisition_run_id": "langgraph-rag-acquisition-015e3d97fc52-370e1038",
    "tree_run_id": "langgraph-rag-tree-015e3d97fc52-42c8480b"
  },
  "tree_quality": {
    "status": "accepted",
    "tree_manifest_pa

### Known Limitations

- Default execution is fixture-first so the notebook stays stable and warning-free in the shared repo environment.
- Local real-PDF mode is available, but a polluted tree is treated as a blocked run, not a successful RAG answer.
- Visual enrichment remains attachment-only and uses the noop multimodal provider in this cookbook.
- Live text requires a real `OPENROUTER_API_KEY`; otherwise the notebook falls back to deterministic grounded formatting.
